In [48]:
# DBTokenizer Demo

# This notebook demonstrates the DBTokenizer on synthetic clinical data for two patients.

# Data classes:
# - demographic: birth date
# - lab: hemoglobin, glucose, creatinine
# - vital: heart rate, blood pressure, SpO2
# - medication: compound, route, frequency, units as text; dose as numeric
# - flowsheet: categorical observations (O2 device, diet, fall risk)
# - admission / discharge: hospital stay boundaries

# We run two configurations:
# 1. BPE + Discrete + Factored: numeric values are binned into discrete tokens, placed as separate tokens after the text token
# 2. BPE + Continuous + Fused: numeric values are z-scored and paired in a parallel float array; concept-mode text+level tokens are fused into single compound tokens

In [49]:
import polars as pl
import numpy as np
from dbtoken import DBTokenizer

In [50]:
## Load Demo Data

In [51]:
df = pl.read_csv(
    "data/demo.csv",
    try_parse_dates=True,
    null_values=[""],
)
print(f"Loaded {df.height} rows for {df['id'].n_unique()} patients")
print(f"Classes: {sorted(df['class'].unique().to_list())}")
df.head(20)

Loaded 106 rows for 2 patients
Classes: ['admission', 'demographic', 'discharge', 'flowsheet', 'lab', 'medication', 'vital']


id,time,class,text_value,numeric_value
i64,datetime[μs],str,str,f64
1,1958-03-15 00:00:00,"""demographic""","""birth_date""",null
1,2024-01-02 08:00:00,"""lab""","""hemoglobin""",13.2
1,2024-01-02 08:00:00,"""lab""","""glucose""",98.0
1,2024-01-02 08:00:00,"""lab""","""creatinine""",0.9
1,2024-01-02 08:05:00,"""vital""","""heart_rate""",78.0
…,…,…,…,…
1,2024-01-15 09:30:00,"""lab""","""creatinine""",1.0
1,2024-01-15 09:35:00,"""vital""","""heart_rate""",82.0
1,2024-01-15 09:35:00,"""vital""","""blood_pressure_systolic""",132.0


In [52]:
# Configuration 1: BPE + Discrete + Factored

# This is the simplest BPE configuration:
# - text_mode_default="auto": classes with many unique text values (like medication) get BPE; low-cardinality classes (lab, vital, flowsheet) get concept tokens
# - num_type="discrete": numeric values are quantile-binned into `<|Q0|>` … `<|Q{n}|>` tokens (or `<|L0|>` … for ≤10 distinct values)
# - num_seq="factored": numeric tokens are emitted as separate tokens after the text token

In [53]:
tok1 = DBTokenizer(
    text_mode_default="auto",
    text_mode_threshold=4,
    num_type="discrete",
    num_seq="factored",
    n_bins=5,
    final_vocab_size=1024,
    state_transitions={
        "inpatient": ["admission"],
        "default": ["discharge"],
    },
    milestone_per_state={"inpatient": "8hr", "default": "week"},
)
tok1.train(df)

Extracting birth dates from data...
  Found birth dates for 2 patients
Resolving per-class text modes...
  4 concept classes, 2 BPE classes, 0 pair-level overrides
Training numeric transforms...
  13 numeric groups: 11 level, 2 bins, 0 scaling
Training time-delta transforms...
  scale '0' time-delta: bins (-)
Building vocabulary...
  BPE vocab size: 617 (236 special/concept, 381 BPE merges)
Training complete.


In [54]:
print("Resolved text modes per class:")
for cls, mode in sorted(tok1.class_text_modes.items()):
    print(f"  {cls:20s} -> {mode}")

Resolved text modes per class:
  admission            -> concept
  discharge            -> concept
  flowsheet            -> bpe
  lab                  -> concept
  medication           -> bpe
  vital                -> concept


In [55]:
ids1, vals1 = tok1.encode(df)
print(f"Token count: {len(ids1)}")
print(f"vals is None (discrete mode): {vals1 is None}")
print()

# Decode back to human-readable tokens
decoded1 = tok1.decode(ids1)
print("First 60 tokens:")
for i, t in enumerate(decoded1[:60]):
    print(f"  [{i:3d}] {t}")

Token count: 530
vals is None (discrete mode): True

First 60 tokens:
  [  0] <|sos|>
  [  1] <|age_65|>
  [  2] <|lab|>
  [  3] hemoglobin
  [  4] <|L7|>
  [  5] <|lab|>
  [  6] glucose
  [  7] <|L2|>
  [  8] <|lab|>
  [  9] creatinine
  [ 10] <|L1|>
  [ 11] <|delta_time_0|>
  [ 12] <|Q0|>
  [ 13] <|ms_week_1|>
  [ 14] <|vital|>
  [ 15] heart_rate
  [ 16] <|Q1|>
  [ 17] <|vital|>
  [ 18] blood_pressure_systolic
  [ 19] <|Q1|>
  [ 20] <|vital|>
  [ 21] blood_pressure_diastolic
  [ 22] <|L3|>
  [ 23] <|vital|>
  [ 24] spo2
  [ 25] <|L5|>
  [ 26] <|delta_time_0|>
  [ 27] <|Q0|>
  [ 28] <|medication|>
  [ 29] lisinopril
  [ 30] ,
  [ 31]  oral
  [ 32] ,
  [ 33]  daily
  [ 34] ,
  [ 35]  mg
  [ 36] <|L0|>
  [ 37] <|medication|>
  [ 38] metformin
  [ 39] ,
  [ 40]  oral
  [ 41] ,
  [ 42]  BID
  [ 43] ,
  [ 44]  mg
  [ 45] <|L0|>
  [ 46] <|delta_time_0|>
  [ 47] <|Q0|>
  [ 48] <|flowsheet|>
  [ 49] o
  [ 50] 2
  [ 51] _device
  [ 52] :
  [ 53]  room
  [ 54] _air
  [ 55] <|flowsheet|>
  [ 56]

In [56]:
# Debug view: see per-row token breakdown
dbg1 = tok1.encode_debug(df)
print("\nDebug view of first 10 rows:")
dbg1.head(10)


Debug view of first 10 rows:


id,time,class,text_value,numeric_value,tokens
i64,datetime[μs],str,str,f64,list[str]
1,1958-03-15 00:00:00,"""demographic""","""birth_date""",null,"[""<|sos|>""]"
1,2024-01-02 08:00:00,"""lab""","""hemoglobin""",13.2,"[""<|age_65|>"", ""<|lab|>"", … ""<|L7|>""]"
1,2024-01-02 08:00:00,"""lab""","""glucose""",98.0,"[""<|lab|>"", ""glucose"", ""<|L2|>""]"
1,2024-01-02 08:00:00,"""lab""","""creatinine""",0.9,"[""<|lab|>"", ""creatinine"", ""<|L1|>""]"
1,2024-01-02 08:05:00,"""vital""","""heart_rate""",78.0,"[""<|delta_time_0|>"", ""<|Q0|>"", … ""<|Q1|>""]"
1,2024-01-02 08:05:00,"""vital""","""blood_pressure_systolic""",128.0,"[""<|vital|>"", ""blood_pressure_systolic"", ""<|Q1|>""]"
1,2024-01-02 08:05:00,"""vital""","""blood_pressure_diastolic""",82.0,"[""<|vital|>"", ""blood_pressure_diastolic"", ""<|L3|>""]"
1,2024-01-02 08:05:00,"""vital""","""spo2""",97.0,"[""<|vital|>"", ""spo2"", ""<|L5|>""]"
1,2024-01-02 08:10:00,"""medication""","""lisinopril, oral, daily, mg""",10.0,"[""<|delta_time_0|>"", ""<|Q0|>"", … ""<|L0|>""]"


In [57]:
# # Configuration 2: BPE + Continuous + Fused

# This configuration demonstrates:
# - num_type="continuous": numeric values are distribution-scaled to z-scores, stored in a parallel `vals` array
# - num_seq="fused": for concept-mode text groups, the text token and its level are combined into a single compound token (e.g. `hemoglobin::L4`)

# ### Important: Concept Overrides for Fused Numerics

# Fused mode only works with concept text tokens (not BPE). For classes that have true numeric measurements you want fused with levels, you must ensure they use concept mode.

# Since our lab and vital classes have low cardinality, `auto` already assigns them concept mode. But if any class were incorrectly assigned BPE, you'd override it:
# text_mode_overrides={"lab": "concept", "vital": "concept"}

# BPE classes automatically fall back to factored numeric placement (with a warning).

In [58]:
tok2 = DBTokenizer(
    text_mode_default="auto",
    text_mode_threshold=4,
    # Explicitly ensure numeric-bearing classes use concept mode for fusion
    text_mode_overrides={
        "lab": "concept",
        "vital": "concept",
        "flowsheet": "concept",
    },
    num_type="continuous",
    num_seq="fused",
    n_bins=5,
    final_vocab_size=1024,
    state_transitions={
        "inpatient": ["admission"],
        "default": ["discharge"],
    },
    milestone_per_state={"inpatient": "8hr", "default": "week"},
)
tok2.train(df)

Extracting birth dates from data...
  Found birth dates for 2 patients
Resolving per-class text modes...
  5 concept classes, 1 BPE classes, 0 pair-level overrides
Training numeric transforms...
  13 numeric groups: 11 level, 0 bins, 2 scaling
Training time-delta transforms...
  scale '0' time-delta: scaling (lognormal)
Building vocabulary...
  BPE vocab size: 615 (285 special/concept, 330 BPE merges)
Training complete.


/Users/andrewloza/Documents/Research/dbtoken/dbtoken/tokenizer.py:605: UserWarning: num_seq='fused' but 1 classes use BPE text mode (e.g. ['medication']). These will silently use factored numeric placement unless overridden at pair level.
  warnings.warn(
/Users/andrewloza/Documents/Research/dbtoken/dbtoken/tokenizer.py:654: UserWarning: Distribution auto-detect failed: unable to find column "count"; valid columns: []. Falling back to normal.
  warnings.warn(


In [59]:
print("Resolved text modes per class:")
for cls, mode in sorted(tok2.class_text_modes.items()):
    print(f"  {cls:20s} -> {mode}")

print()
print("Numeric parameter types:")
for key, params in sorted(tok2.numeric_params.items(), key=str):
    cls, tv = key
    print(f"  ({cls}, {tv}): {params['type']}")

Resolved text modes per class:
  admission            -> concept
  discharge            -> concept
  flowsheet            -> concept
  lab                  -> concept
  medication           -> bpe
  vital                -> concept

Numeric parameter types:
  (lab, creatinine): level
  (lab, glucose): level
  (lab, hemoglobin): level
  (medication, atorvastatin, oral, daily, mg): level
  (medication, furosemide, IV, BID, mg): level
  (medication, heparin, IV, continuous, units): level
  (medication, lisinopril, oral, daily, mg): level
  (medication, metformin, oral, BID, mg): level
  (medication, vancomycin, IV, q12h, mg): level
  (vital, blood_pressure_diastolic): level
  (vital, blood_pressure_systolic): scaling
  (vital, heart_rate): scaling
  (vital, spo2): level


In [60]:
ids2, vals2 = tok2.encode(df)
print(f"Token count: {len(ids2)}")
print(f"vals length:  {len(vals2)}")
print()

# Show tokens alongside their continuous values
decoded2 = tok2.decode(ids2)
print("First 60 token-value pairs:")
for i, (t, v) in enumerate(zip(decoded2[:60], vals2[:60])):
    v_str = f"{v:8.3f}" if not np.isnan(v) else "     NaN"
    print(f"  [{i:3d}] {v_str}  {t}")

Token count: 374
vals length:  374

First 60 token-value pairs:
  [  0]      NaN  <|sos|>
  [  1]      NaN  <|age_65|>
  [  2]      NaN  <|lab|>
  [  3]      NaN  hemoglobin::L7
  [  4]      NaN  <|lab|>
  [  5]      NaN  glucose::L2
  [  6]      NaN  <|lab|>
  [  7]      NaN  creatinine::L1
  [  8]   -0.996  <|delta_time_0|>
  [  9]      NaN  <|ms_week_1|>
  [ 10]      NaN  <|vital|>
  [ 11]   -0.557  heart_rate
  [ 12]      NaN  <|vital|>
  [ 13]   -0.555  blood_pressure_systolic
  [ 14]      NaN  <|vital|>
  [ 15]      NaN  blood_pressure_diastolic::L3
  [ 16]      NaN  <|vital|>
  [ 17]      NaN  spo2::L5
  [ 18]   -0.996  <|delta_time_0|>
  [ 19]      NaN  <|medication|>
  [ 20]      NaN  lisinopril
  [ 21]      NaN  ,
  [ 22]      NaN   oral
  [ 23]      NaN  ,
  [ 24]      NaN   daily
  [ 25]      NaN  ,
  [ 26]      NaN   mg
  [ 27]      NaN  <|L0|>
  [ 28]      NaN  <|medication|>
  [ 29]      NaN  metformin
  [ 30]      NaN  ,
  [ 31]      NaN   oral
  [ 32]      NaN  ,
  [ 3

In [61]:
# Debug view: fused tokens
dbg2 = tok2.encode_debug(df)
dbg2

id,time,class,text_value,numeric_value,tokens
i64,datetime[μs],str,str,f64,list[str]
1,1958-03-15 00:00:00,"""demographic""","""birth_date""",null,"[""<|sos|>""]"
1,2024-01-02 08:00:00,"""lab""","""hemoglobin""",13.2,"[""<|age_65|>"", ""<|lab|>"", ""hemoglobin::L7""]"
1,2024-01-02 08:00:00,"""lab""","""glucose""",98.0,"[""<|lab|>"", ""glucose::L2""]"
1,2024-01-02 08:00:00,"""lab""","""creatinine""",0.9,"[""<|lab|>"", ""creatinine::L1""]"
1,2024-01-02 08:05:00,"""vital""","""heart_rate""",78.0,"[""<|delta_time_0|>(-0.996)"", ""<|ms_week_1|>"", … ""heart_rate(-0.557)""]"
…,…,…,…,…,…
2,2024-03-20 10:15:00,"""vital""","""blood_pressure_systolic""",122.0,"[""<|vital|>"", ""blood_pressure_systolic(-1.160)""]"
2,2024-03-20 10:15:00,"""vital""","""blood_pressure_diastolic""",78.0,"[""<|vital|>"", ""blood_pressure_diastolic::L1""]"
2,2024-03-20 10:15:00,"""vital""","""spo2""",98.0,"[""<|vital|>"", ""spo2::L6""]"


## Comparison

Let's compare the two configurations side by side.

In [62]:
print(f"Configuration 1 (BPE + Discrete + Factored):")
print(f"  Total tokens: {len(ids1)}")
print(f"  Vocab size:   {tok1.final_vocab_size}")
print(f"  vals array:   {'None' if vals1 is None else len(vals1)}")
print()
print(f"Configuration 2 (BPE + Continuous + Fused):")
print(f"  Total tokens: {len(ids2)}")
print(f"  Vocab size:   {tok2.final_vocab_size}")
print(f"  vals array:   {len(vals2)}")
print()
print(f"Fused mode reduces sequence length by "
      f"{len(ids1) - len(ids2)} tokens ({(1 - len(ids2)/len(ids1))*100:.1f}%)")

Configuration 1 (BPE + Discrete + Factored):
  Total tokens: 530
  Vocab size:   1024
  vals array:   None

Configuration 2 (BPE + Continuous + Fused):
  Total tokens: 374
  Vocab size:   1024
  vals array:   374

Fused mode reduces sequence length by 156 tokens (29.4%)


## Save & Load Round-Trip

In [63]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "demo_tokenizer")
    tok2.save(path)
    tok2_loaded = DBTokenizer.load(path)

    ids2_rt, vals2_rt = tok2_loaded.encode(df)
    assert ids2 == ids2_rt, "Token ID mismatch after round-trip!"
    assert len(vals2) == len(vals2_rt), "Vals length mismatch!"
    print("Save/load round-trip: PASSED ✓")

Saved tokenizer to /var/folders/jx/qpq3p10j7nz660bbfmr0pkrm0000gn/T/tmp1byzv10a/demo_tokenizer.json
Loaded tokenizer from /var/folders/jx/qpq3p10j7nz660bbfmr0pkrm0000gn/T/tmp1byzv10a/demo_tokenizer.json
Save/load round-trip: PASSED ✓


In [64]:
# now we will turn the token stream back into a dataframe and see how it compares to the original

df_decoded = tok2.decode_to_dataframe(ids2, vals2)
print("Decoded dataframe head:")
print(df_decoded.head(10))    
print("\nOriginal dataframe head:")
print(df.head(10))

Decoded dataframe head:
shape: (10, 5)
┌─────┬─────────────────────┬────────────┬─────────────────────────────┬───────────────┐
│ id  ┆ time                ┆ class      ┆ text_value                  ┆ numeric_value │
│ --- ┆ ---                 ┆ ---        ┆ ---                         ┆ ---           │
│ i64 ┆ datetime[μs]        ┆ str        ┆ str                         ┆ f64           │
╞═════╪═════════════════════╪════════════╪═════════════════════════════╪═══════════════╡
│ 0   ┆ 1970-01-01 00:00:00 ┆ lab        ┆ hemoglobin                  ┆ 13.2          │
│ 0   ┆ 1970-01-01 00:00:00 ┆ lab        ┆ glucose                     ┆ 98.0          │
│ 0   ┆ 1970-01-01 00:05:00 ┆ lab        ┆ creatinine                  ┆ 0.9           │
│ 0   ┆ 1970-01-01 00:05:00 ┆ vital      ┆ heart_rate                  ┆ 78.0          │
│ 0   ┆ 1970-01-01 00:05:00 ┆ vital      ┆ blood_pressure_systolic     ┆ 128.0         │
│ 0   ┆ 1970-01-01 00:05:00 ┆ vital      ┆ blood_pressure_diastolic    

## Plain Concept vs. Split Concept vs. BPE Tokenization

The configurations above remain unchanged. This isolated example contrasts one-token concept values, prescribed per-class splitting, and learned BPE pieces. Medication values use the literal delimiter `", "` (comma followed by a space), while vital values use `"_"`.

In [65]:
# Select one representative row from each class so the contrast is easy to see.
split_examples = pl.concat([
    df.filter(pl.col("class") == "vital").head(1),
    df.filter(pl.col("class") == "medication").head(1),
]).sort(["id", "time", "class"])
split_examples.select(["class", "text_value", "numeric_value"])

class,text_value,numeric_value
str,str,f64
"""vital""","""heart_rate""",78.0
"""medication""","""lisinopril, oral, daily, mg""",10.0


In [86]:
plain_concept_tok = DBTokenizer(
    text_mode_default="concept",
    num_type="discrete",
    num_seq="factored",
)
split_concept_tok = DBTokenizer(
    text_mode_default="concept",
    concept_splitters={
        "medication": ", ",
        "vital": "_",
    },
    num_type="discrete",
    num_seq="factored",
)
bpe_tok = DBTokenizer(
    text_mode_default="bpe",
    num_type="discrete",
    num_seq="factored",
    final_vocab_size=510,
)

plain_concept_tok.train(split_examples)
split_concept_tok.train(split_examples)
bpe_tok.train(split_examples)

Extracting birth dates from data...
  No birth-date rows found
Resolving per-class text modes...
  2 concept classes, 0 BPE classes, 0 pair-level overrides
Training numeric transforms...
  2 numeric groups: 2 level, 0 bins, 0 scaling
Training time-delta transforms...
  scale '0' time-delta: bins (-)
Building vocabulary...
  Concept vocab size: 230
Training complete.
Extracting birth dates from data...
  No birth-date rows found
Resolving per-class text modes...
  2 concept classes, 0 BPE classes, 0 pair-level overrides
Training numeric transforms...
  2 numeric groups: 2 level, 0 bins, 0 scaling
Training time-delta transforms...
  scale '0' time-delta: bins (-)
Building vocabulary...
  Concept vocab size: 234
Training complete.
Extracting birth dates from data...
  No birth-date rows found
Resolving per-class text modes...
  0 concept classes, 2 BPE classes, 0 pair-level overrides
Training numeric transforms...
  2 numeric groups: 2 level, 0 bins, 0 scaling
Training time-delta transfor

In [90]:
plain_debug = plain_concept_tok.encode_debug(split_examples)
split_debug = split_concept_tok.encode_debug(split_examples)
bpe_debug = bpe_tok.encode_debug(split_examples)

# replace double pipe with quote comma quote. no surrouting spaces on the tokens

# print original data rows
print(split_examples)

print("Demonstration of concept splitting using \"_\" for vitals and \", \" for meds as the splitters")

for row, plain_tokens, split_tokens, bpe_tokens in zip(
    split_examples.iter_rows(named=True),
    plain_debug["tokens"].to_list(),
    split_debug["tokens"].to_list(),
    bpe_debug["tokens"].to_list(),
):
    print(f"{row['class'].upper()}: {row['text_value']}")
    print("  plain concept:")
    print("   " + "\", \"".join(plain_tokens))
    print("  split concept:")
    print("   " + "\", \"".join(split_tokens))
    print("  BPE:")
    print("   " + "\", \"".join(bpe_tokens))
    print()

shape: (2, 5)
┌─────┬─────────────────────┬────────────┬─────────────────────────────┬───────────────┐
│ id  ┆ time                ┆ class      ┆ text_value                  ┆ numeric_value │
│ --- ┆ ---                 ┆ ---        ┆ ---                         ┆ ---           │
│ i64 ┆ datetime[μs]        ┆ str        ┆ str                         ┆ f64           │
╞═════╪═════════════════════╪════════════╪═════════════════════════════╪═══════════════╡
│ 1   ┆ 2024-01-02 08:05:00 ┆ vital      ┆ heart_rate                  ┆ 78.0          │
│ 1   ┆ 2024-01-02 08:10:00 ┆ medication ┆ lisinopril, oral, daily, mg ┆ 10.0          │
└─────┴─────────────────────┴────────────┴─────────────────────────────┴───────────────┘
Demonstration of concept splitting using "_" for vitals and ", " for meds as the splitters
VITAL: heart_rate
  plain concept:
   <|sos|>", "<|vital|>", "heart_rate", "<|L0|>
  split concept:
   <|sos|>", "<|vital|>", "heart", "rate", "<|L0|>
  BPE:
   <|sos|>", "<|vital|>",

## MEDS-Style Hierarchical Codes

MEDS-style codes often carry an explicit hierarchy. Here each code uses a literal double backslash (`\\`) between levels, so the tokenizer can preserve those prescribed semantic boundaries instead of learning them from text.

In [ ]:
from datetime import datetime, timedelta

meds_examples = pl.DataFrame(
    [
        {
            "id": 1,
            "time": datetime(2024, 1, 1),
            "class": "meds_code",
            "text_value": r"LAB\\LOINC\\2345-7",
            "numeric_value": None,
        },
        {
            "id": 1,
            "time": datetime(2024, 1, 1) + timedelta(minutes=1),
            "class": "meds_code",
            "text_value": r"DIAGNOSIS\\ICD10CM\\E11.9",
            "numeric_value": None,
        },
    ],
    schema_overrides={"numeric_value": pl.Float64},
)

meds_split_tok = DBTokenizer(
    text_mode_default="auto",
    concept_splitters={"meds_code": r"\\"},
    num_type="discrete",
    num_seq="factored",
)
meds_split_tok.train(meds_examples)
meds_debug = meds_split_tok.encode_debug(meds_examples)

for code, tokens in zip(
    meds_debug["text_value"].to_list(),
    meds_debug["tokens"].to_list(),
):
    pieces = [token for token in tokens if not token.startswith("<|")]
    print(f"MEDS code: {code!r}")
    print(f"  concept pieces: {pieces}")

meds_ids, meds_vals = meds_split_tok.encode(meds_examples)
meds_reconstructed = meds_split_tok.decode_to_dataframe(
    meds_ids, meds_vals, reference_time=datetime(2024, 1, 1)
)
assert meds_reconstructed["text_value"].to_list() == meds_examples["text_value"].to_list()
print("Exact MEDS code round-trip: PASSED")